# Result analysis

## Usage data extraction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# Model keys
class MK:
    TP = "Thread pool"
    VT = "Virtual threads"
    RX = "Reactive"


class Column:
    ELAPSED = "elapsed_seconds"
    CPU = "cpu_percentage"
    STACK = "stack_committed_mb"
    HEAP = "heap_used_mb"
    GCT = "gct"


thread_pool = pd.read_csv("thread-pool/cpu-memory-usage.csv", skipinitialspace=True)
virtual_thread = pd.read_csv("virtual-thread/cpu-memory-usage.csv", skipinitialspace=True)
reactive = pd.read_csv("reactive/cpu-memory-usage.csv", skipinitialspace=True)

raw_data = {
    MK.TP: {"data": thread_pool, "starts": {100: 167, 200: 782, 500: 1398}},
    MK.VT: {"data": virtual_thread, "starts": {100: 146, 200: 761, 500: 1376}},
    MK.RX: {"data": reactive, "starts": {100: 147, 200: 761, 500: 1377}}
}

test_delays = [100, 200, 500]

time_span_per_test = 600

num_tests_per_model = 4

total_time_span = time_span_per_test * num_tests_per_model

window_size_for_plot = int(time_span_per_test / 2)

window_start_for_plot = int(time_span_per_test / 4)

In [ ]:
def extract_column(df, start_sec, col,  multiply=1):

    # Make time start from 0
    df = df[df[Column.ELAPSED] >= start_sec]
    time_col = df[Column.ELAPSED] - start_sec
    col = df[col]

    # Build continuous timed record
    time_max = list(time_col)[-1]
    result = [None] * (time_max + 1)
    for time, col_val in zip(time_col, col):
        result[time] = col_val

    # Fill None
    result = pd.Series(result).ffill().bfill()
    if multiply != 1:
        result = result * multiply
    return result.tolist()


def avg(values):
    return round(sum(values) / len(values), 1)


def span(values):
    return round(values[-1] - values[0], 6)


def extract_results(raw_data, col_name,
                 summary_avg=False, summary_change=False, multiply=1):

    combined_data = {}
    summary = {}
    for delay in test_delays:

        col_tp = extract_column(raw_data[MK.TP]["data"], raw_data[MK.TP]["starts"][delay], col_name, multiply)
        col_vt = extract_column(raw_data[MK.VT]["data"], raw_data[MK.VT]["starts"][delay], col_name, multiply)
        col_rx = extract_column(raw_data[MK.RX]["data"], raw_data[MK.RX]["starts"][delay], col_name, multiply)

        sam_start = window_start_for_plot
        sam_end = sam_start + window_size_for_plot

        sample_tp = col_tp[sam_start: sam_end]
        sample_vt = col_vt[sam_start: sam_end]
        sample_rx = col_rx[sam_start: sam_end]

        if summary_avg:
            summary[delay] = {
                col_name: {
                    MK.TP: avg(sample_tp),
                    MK.VT: avg(sample_vt),
                    MK.RX: avg(sample_rx)
                }
            }

        if summary_change:
            summary[delay] = {
                col_name: {
                    MK.TP: span(sample_tp),
                    MK.VT: span(sample_vt),
                    MK.RX: span(sample_rx)
                }
            }

        combined_data[delay] = {
            MK.TP: sample_tp,
            MK.VT: sample_vt,
            MK.RX: sample_rx
        }

    return combined_data, summary


cpu_data, cpu_summary = extract_results(raw_data, Column.CPU, summary_avg = True)
stack_data, stack_summary = extract_results(raw_data, Column.STACK, summary_avg = True)
heap_data, heap_summary = extract_results(raw_data, Column.HEAP, summary_avg = True)
gct_data, gct_summary = extract_results(raw_data, Column.GCT, summary_change = True, multiply=1000)
ygc_data, ygc_summary = extract_results(raw_data, "ygc", summary_change = True)

fgc_data, fgc_summary = extract_results(raw_data, "fgc", summary_change = True)

cgc_data, cgc_summary = extract_results(raw_data, "cgc", summary_change = True)

ts_data, ts_summary = extract_results(raw_data, "threads", summary_avg = True)

summary = {}
for delay in test_delays:
    summary[delay] = (cpu_summary[delay] |
                      ts_summary[delay] |
                      stack_summary[delay] |
                      heap_summary[delay] |
                      ygc_summary[delay] |
                      fgc_summary[delay] |
                      cgc_summary[delay] |
                      gct_summary[delay]
                      )
    df = pd.DataFrame(summary[delay])
    df.columns = ["CPU (%)", "threads", "Stack (MB)",
                  "Heap (MB)", "YGC", "FGC", "CGC", "GCT (ms)"]
    print(f'\n--- Resource utilization metrics under {delay} ms delay ({window_size_for_plot}-second sample) ---')
    print(df.to_latex(float_format="{:,.1f}".format))
    # print(df.to_latex())


## CPU plot

In [ ]:
def draw_cpu_plot(delays, rows=1, columns=1, figsize=(9,6)):
    # Configuration for colors and labels
    models = {
    'Thread pool': {'color': '#1f77b4', 'linestyle': '-'},
    'Virtual threads': {'color': '#2ca02c', 'linestyle': '-'},
    'Reactive streams': {'color': '#ff7f0e', 'linestyle': '--'} # Dashed to distinguish from VT
}

    # 1. Setup the figure and grid (2 rows, 2 columns)
    fig, axes = plt.subplots(rows, columns, figsize=figsize, sharex=True, squeeze=False)
    axes = axes.flatten()

    # 2. Loop through each delay to create a subplot
    for i, delay in enumerate(delays):
        ax = axes[i]

        # --- DATA PLUG-IN START ---
        time_sec = np.arange(window_size_for_plot)

        # Mock data for demonstration - replace with your actual CPU records
        tp_data = cpu_data[delay][MK.TP]
        vt_data = cpu_data[delay][MK.VT]
        rx_data = cpu_data[delay][MK.RX]

        ax.plot(time_sec, tp_data, label='Thread pool', **models['Thread pool'])
        ax.plot(time_sec, vt_data, label='Virtual threads', **models['Virtual threads'])
        ax.plot(time_sec, rx_data, label='Reactive streams', **models['Reactive streams'])
        # --- DATA PLUG-IN END ---

        # Subplot Styling
        if len(delays) > 1:
            ax.set_title(f'Simulated delay: {delay}ms', fontsize=12)
            ax.grid(True, linestyle=':', alpha=0.6)

        # Axis Labels (Only on the outer edges for cleanliness)
        if i >= 2 or len(delays) == 1:
            ax.set_xlabel('Time (seconds)', fontsize=11)
        if i % 2 == 0:
            ax.set_ylabel('CPU usage (%)', fontsize=11)

    # 3. Global Figure Styling
    if len(delays) > 1:
        delay_title = 'variable I/O latency'
    else:
        delay_title = f'{delays[0]} ms delay'

    plt.suptitle(f'CPU utilization across JVM concurrency models under {delay_title}',
                 fontsize=14, y=0.98)

    # Create a single legend at the top
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.94),
               ncol=3, frameon=False, fontsize=12)

    # Adjust layout to make room for titles and legend
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])

    # 4. Save and Show
    plt.savefig(f'cpu_metrics_{'_'.join([str(x) for x in delays])}.png', dpi=300, bbox_inches='tight')
    plt.show()


draw_cpu_plot([100])
draw_cpu_plot([200])
draw_cpu_plot([500])

## Memory plot

In [ ]:
def draw_memory_plot(delays, figsize):
    # Set figure size for a vertical layout
    fig, axes = plt.subplots(len(delays), 2, figsize=figsize, sharex=True, squeeze=False)

    colors = {'TP': '#1f77b4', 'VT': '#2ca02c', 'RX': '#ff7f0e'}
    labels = {'TP': 'Thread pool', 'VT': 'Virtual threads', 'RX': 'Reactive streams'}

    for i, delay in enumerate(delays):
        # Left Column: Heap Used
        ax_h = axes[i, 0]
        ax_h.plot(time_sec, heap_data[delay][MK.TP], color=colors['TP'], label=labels['TP']) # Replace with actual data
        ax_h.plot(time_sec, heap_data[delay][MK.VT], color=colors['VT'], label=labels['VT'])
        ax_h.plot(time_sec, heap_data[delay][MK.RX], color=colors['RX'], label=labels['RX'], ls='--')

        ax_h.set_title(f'Heap used')
        ax_h.set_ylabel('MB')
        ax_h.grid(True, alpha=0.3)

        # Right Column: Stack Reserved
        ax_s = axes[i, 1]
        ax_s.plot(time_sec, stack_data[delay][MK.TP], color=colors['TP'])
        ax_s.plot(time_sec, stack_data[delay][MK.VT], color=colors['VT'])
        ax_s.plot(time_sec, stack_data[delay][MK.RX], color=colors['RX'], ls='--')

        ax_s.set_title(f'Stack committed')
        ax_s.set_ylabel('MB')
        ax_s.grid(True, alpha=0.3)

        ax_h.set_xlabel('Time (seconds)')
        ax_s.set_xlabel('Time (seconds)')

    # Global Legend
    handles, labs = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labs, loc='upper center', bbox_to_anchor=(0.5, 0.99),
               ncol=3, frameon=False, fontsize=12)
    plt.suptitle(f'Memory utilization across JVM concurrency models under {delays[0]} ms delay',
                 fontsize=14, y=1.05)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f"memory_metrics_{'_'.join([str(x) for x in delays])}.png", dpi=300, bbox_inches='tight')
    plt.show()


draw_memory_plot([100], (12, 6))
draw_memory_plot([200], (12, 6))
draw_memory_plot([500], (12, 6))


## Throughput and latency data extraction

In [ ]:
import pandas as pd
import re
import os

def parse_time_to_ms(time_str):
    """Converts time strings like '1.08s' or '281.61ms' to a float in milliseconds."""
    if not time_str:
        return None
    time_str = time_str.strip().lower()

    if time_str.endswith('ms'):
        return float(time_str.replace('ms', ''))
    elif time_str.endswith('us'):
        return float(time_str.replace('us', '')) / 1000.0
    elif time_str.endswith('s'):
        return float(time_str.replace('s', '')) * 1000.0
    elif time_str.endswith('m'):
        return float(time_str.replace('m', '')) * 60000.0

    return float(time_str)

def extract_wrk_data(file_content, model_name):
    """Parses wrk output and extracts metrics for the 10-minute test runs."""
    results = []

    # Split the document by test runs
    runs = re.split(r'Running \d+m test @ ', file_content)[2:] # Skip warm-up

    for run in runs:
        # Extract the delay from the URL path
        delay_match = re.search(r'/delay/(\d+)', run)
        if not delay_match:
            continue
        delay = int(delay_match.group(1))

        # Extract Avg and Max Latency
        # Format: Latency   324.22ms  144.57ms   2.04s    91.60%
        lat_stats = re.search(r'Latency\s+([\d\.]+[a-zA-Z]+)\s+([\d\.]+[a-zA-Z]+)\s+([\d\.]+[a-zA-Z]+)', run)
        avg_lat = parse_time_to_ms(lat_stats.group(1)) if lat_stats else None
        max_lat = parse_time_to_ms(lat_stats.group(3)) if lat_stats else None

        # Extract Latency Percentiles
        p50 = re.search(r'50%\s+([\d\.]+[a-zA-Z]+)', run)
        p99 = re.search(r'99%\s+([\d\.]+[a-zA-Z]+)', run)
        p50_lat = parse_time_to_ms(p50.group(1)) if p50 else None
        p99_lat = parse_time_to_ms(p99.group(1)) if p99 else None

        # Extract Throughput
        req_sec = re.search(r'Requests/sec:\s+([\d\.]+)', run)
        throughput = float(req_sec.group(1)) if req_sec else None

        results.append({
            'Model': model_name,
            'Delay': delay,
            'Requests/sec': throughput,
            'Avg latency (ms)': round(avg_lat, 2) if avg_lat else None,
            'P50 latency (ms)': round(p50_lat, 2) if p50_lat else None,
            'P99 latency (ms)': round(p99_lat, 2) if p99_lat else None,
            'Max latency (ms)': round(max_lat, 2) if max_lat else None
        })

    return results

# Map of the models to their respective file paths
files_to_process = {
    'Thread pool': 'thread-pool/throughput-latency.txt',
    'Virtual threads': 'virtual-thread/throughput-latency.txt',
    'Reactive': 'reactive/throughput-latency.txt'
}

all_parsed_data = []

# Iterate through the files and extract data
for model_name, filepath in files_to_process.items():
    try:
        with open(filepath, 'r') as f:
            content = f.read()
            parsed_runs = extract_wrk_data(content, model_name)
            all_parsed_data.extend(parsed_runs)
    except FileNotFoundError:
        print(f"Warning: Could not find file {filepath}. Make sure the script is in the correct directory.")

# Convert to a master DataFrame
master_df = pd.DataFrame(all_parsed_data)

# Dictionary to hold the separated DataFrames for each delay
dfs_by_delay = {}

if not master_df.empty:
    for delay in [100, 200, 500]:
        # Filter by delay and drop the Delay column as it's no longer needed in the final table
        delay_df = master_df[master_df['Delay'] == delay].drop(columns=['Delay'])

        # Set 'Model' as the index for clean row labels
        delay_df = delay_df.set_index('Model')

        dfs_by_delay[delay] = delay_df

        # Print the resulting DataFrame
        print(f"\n--- DataFrame for {delay} ms delay ---")
        print(delay_df.to_latex(float_format="{:,.1f}".format))
